In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
df = pd.read_csv('data/final.csv')

In [ ]:
df

In [ ]:
last_date = df["DATE"].max()


tickers = {
    "DXY Index": "DX-Y.NYB",
    "LUATTRUU Index": "AGG",
    "NDDUUS Index": "URTH",
    "VIX Index": "^VIX",
    "XAU Curncy": "GC=F",
    "sp500": "^GSPC"
}

raw = yf.download(
    list(tickers.values()),
    start="2025-07-01",
    auto_adjust=True
)

# 🔑 jede Serie explizit holen (kein MultiIndex-Fehler mehr)
data = pd.DataFrame({
    name: raw["Close"][ticker]
    for name, ticker in tickers.items()
})

# Monatsende
data_eom = data.resample("ME").last().reset_index()
data_eom.rename(columns={"Date": "DATE"}, inplace=True)

In [ ]:
# letzter Bloomberg-Wert
last_gold = df.loc[df["DATE"] == df["DATE"].max(), "XAU Curncy"].values[0]

# erster neuer Wert
first_gold = data_eom["XAU Curncy"].dropna().iloc[0]

# Skalierung
scale_gold = last_gold / first_gold

# anwenden
data_eom["XAU Curncy"] = data_eom["XAU Curncy"] * scale_gold

In [ ]:
data_eom

In [ ]:
# letzter Bloomberg-Wert
last_old = df.loc[df["DATE"] == df["DATE"].max(), "LUATTRUU Index"].values[0]

# erster neuer AGG-Wert (nicht NaN)
first_new = data_eom["LUATTRUU Index"].dropna().iloc[0]

# Skalierungsfaktor
scale = last_old / first_new

# AGG anpassen
data_eom["LUATTRUU Index"] = data_eom["LUATTRUU Index"] * scale

In [ ]:
data_eom

In [ ]:
data_eom["VIX Index"].plot()

In [ ]:
# -----------------------------
# 1. Beide DATE-Spalten als datetime
# -----------------------------
df["DATE"] = pd.to_datetime(df["DATE"])
data_eom["DATE"] = pd.to_datetime(data_eom["DATE"])

# -----------------------------
# 2. Alten Juli entfernen (weil jetzt EOM vorhanden)
# -----------------------------
df_clean = df[df["DATE"] < "2025-07-31"]

# -----------------------------
# 3. Spalten angleichen
# -----------------------------
# Unnamed: 0 rauswerfen (optional)
df_clean = df_clean.drop(columns=["Unnamed: 0"], errors="ignore")

# -----------------------------
# 4. Zusammenführen
# -----------------------------
df_extended = pd.concat([df_clean, data_eom], ignore_index=True)

# -----------------------------
# 5. Sortieren
# -----------------------------
df_extended = df_extended.sort_values("DATE").reset_index(drop=True)

print(df_extended.tail(20))

In [ ]:
df_extended = df_extended[df_extended["DATE"] != "2025-07-01"]

In [ ]:
df_extended.tail(20)

In [ ]:
df_extended.to_csv("data/updated.csv")